# Lastmessung auf dem Hub

Für mich, nicht zum Austeilen. Misst, was die Einheiten in der Stunde wirklich rechnen, einmal
auf CPU und, wenn der Hub eine zuteilt, auf der GPU. Ergebnis landet als JSON in
`~/lastmessung/`, mit Rechnername und Uhrzeit im Dateinamen.

Zwei Durchgänge:

1. **Allein.** Einmal starten, wenn niemand sonst rechnet. Das ist die Grundlinie.
2. **Generalprobe.** Zwanzig Sitzungen gleichzeitig öffnen und überall "Run All" drücken. Danach
   die JSON-Dateien einsammeln und die Zeiten mit der Grundlinie vergleichen. Erst das sagt, ob
   der Hub zwanzig Kernel trägt; eine Sitzung allein sagt nur, was ein Kernel kostet.

Die Notebooks der Einheiten rechnen alle auf CPU, kein `.to(device)` irgendwo. Die GPU-Zeilen
hier sagen also, was ein Umbau bringen würde, nicht was in der Stunde passiert.

In [ ]:
import json, os, platform, socket, sys, time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

sys.path.insert(0, "../data")
import diffusion as D


def lies(pfad):
    try:
        return Path(pfad).read_text().strip()
    except OSError:
        return None


umgebung = dict(
    rechner=socket.gethostname(),
    python=platform.python_version(),
    torch=torch.__version__,
    cuda=torch.cuda.is_available(),
    gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    cpu_sichtbar=os.cpu_count(),
    cpu_zugeteilt=len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else None,
    cpu_quote=lies("/sys/fs/cgroup/cpu.max"),
    speicher_grenze=lies("/sys/fs/cgroup/memory.max"),
    torch_threads=torch.get_num_threads(),
)
for k, v in umgebung.items():
    print(f"{k:16s} {v}")

geraete = ["cpu"] + (["cuda"] if torch.cuda.is_available() else [])
ergebnis = {}


def stoppe(name, geraet, fn):
    if geraet == "cuda":
        torch.cuda.synchronize()
    t = time.perf_counter()
    fn(geraet)
    if geraet == "cuda":
        torch.cuda.synchronize()
    s = time.perf_counter() - t
    ergebnis.setdefault(name, {})[geraet] = round(s, 2)
    print(f"{name:40s} {geraet:5s} {s:7.1f} s")

## Einheit 2 und 3: kleine Netze, viele Schritte

Einheit 2 trainiert ein Netz mit 16 Neuronen 2000 Schritte. Einheit 3 trainiert vier Breiten je
6000 Schritte und legt alle 25 Schritte eine Kopie der Gewichte ab; das ist die längste Rechnung
in der ersten Hälfte des Kurses.

In [ ]:
df = pd.read_csv("../data/gebaeude_wien.csv")


def einheit2(geraet):
    kx = np.sort(df["kompaktheit"].to_numpy())
    ky = 40 + 9 * kx + 45 * np.sin(0.8 * kx)
    X = torch.tensor((kx - kx.mean()) / kx.std(), dtype=torch.float32, device=geraet).unsqueeze(1)
    Y = torch.tensor((ky - ky.mean()) / ky.std(), dtype=torch.float32, device=geraet).unsqueeze(1)
    torch.manual_seed(0)
    m = nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 1)).to(geraet)
    opt = torch.optim.Adam(m.parameters(), lr=0.05)
    for _ in range(2000):
        f = nn.functional.mse_loss(m(X), Y)
        opt.zero_grad()
        f.backward()
        opt.step()


def einheit3(geraet):
    zufall = np.random.default_rng(7)
    k = zufall.choice(df["kompaktheit"].to_numpy(), size=265, replace=False)
    y = 40 + 9 * k + 45 * np.sin(0.8 * k) + zufall.normal(0, 20, size=265)
    i = np.arange(265)
    zufall.shuffle(i)
    tr, va = i[:25], i[25:105]
    t = lambda w, m, s: torch.tensor((w - m) / s, dtype=torch.float32, device=geraet).unsqueeze(1)
    Xtr, Ytr = t(k[tr], k[tr].mean(), k[tr].std()), t(y[tr], y[tr].mean(), y[tr].std())
    Xva, Yva = t(k[va], k[tr].mean(), k[tr].std()), t(y[va], y[tr].mean(), y[tr].std())
    for breite in (2, 4, 16, 64):
        torch.manual_seed(0)
        m = nn.Sequential(nn.Linear(1, breite), nn.ReLU(), nn.Linear(breite, breite), nn.ReLU(),
                          nn.Linear(breite, 1)).to(geraet)
        opt = torch.optim.Adam(m.parameters(), lr=0.01)
        for s in range(6000):
            f = nn.functional.mse_loss(m(Xtr), Ytr)
            opt.zero_grad()
            f.backward()
            opt.step()
            if (s + 1) % 25 == 0:
                with torch.no_grad():
                    float(nn.functional.mse_loss(m(Xva), Yva))
                    {n: p.detach().clone() for n, p in m.state_dict().items()}


for g in geraete:
    stoppe("E2  16 Neuronen, 2000 Schritte", g, einheit2)
    stoppe("E3  vier Breiten, je 6000 Schritte", g, einheit3)

## Einheit 6: das Zeichenmodell

3000 Schritte, Stapel 512, alle 100 Schritte die Validierung über alle Fenster. Die Planung
rechnet mit zwei Minuten; das Notebook behauptet zehn Sekunden.

In [ ]:
namen = [z.strip() for z in open("../data/strassennamen.txt", encoding="utf-8") if z.strip()]
zeichen = sorted(set("".join(namen)) | {"^", "$"})
zu_zahl = {z: i for i, z in enumerate(zeichen)}


def fenster(liste):
    X, Y = [], []
    for name in liste:
        folge = [zu_zahl["^"]] * 8 + [zu_zahl[c] for c in name] + [zu_zahl["$"]]
        for i in range(8, len(folge)):
            X.append(folge[i - 8:i])
            Y.append(folge[i])
    return torch.tensor(X), torch.tensor(Y)


def einheit6(geraet):
    i = np.random.default_rng(5).permutation(len(namen))
    Xtr, Ytr = (t.to(geraet) for t in fenster([namen[j] for j in i[:6000]]))
    Xva, Yva = (t.to(geraet) for t in fenster([namen[j] for j in i[6000:]]))
    torch.manual_seed(0)
    m = nn.Sequential(nn.Embedding(len(zeichen), 16), nn.Flatten(), nn.Linear(128, 256),
                      nn.ReLU(), nn.Linear(256, len(zeichen))).to(geraet)
    opt = torch.optim.Adam(m.parameters(), lr=2e-3)
    for s in range(3000):
        wahl = torch.randint(0, len(Xtr), (512,), device=geraet)
        f = nn.functional.cross_entropy(m(Xtr[wahl]), Ytr[wahl])
        opt.zero_grad()
        f.backward()
        opt.step()
        if (s + 1) % 100 == 0:
            with torch.no_grad():
                float(nn.functional.cross_entropy(m(Xva), Yva))
                {n: p.detach().clone() for n, p in m.state_dict().items()}


for g in geraete:
    stoppe("E6  Zeichenmodell, 3000 Schritte", g, einheit6)

## Einheit 7: Ziehen und LoRA

Die Planung fragt nach 200 Schritten mal zwölf Bilder. Gemessen werden das, der Regler mit allen
sechs Schrittzahlen, wie ihn das Notebook vorrechnet, und das LoRA-Training aus der Hausübung.

In [ ]:
Z = torch.tensor(np.load("../data/vae_codes.npz")["code"], dtype=torch.float32)
alpha_quer = D.plan()


def basis(geraet):
    m = D.Rauschschaetzer()
    m.load_state_dict(torch.load("../data/diffusion.pt", map_location="cpu"))
    return m.to(geraet).eval()


@torch.no_grad()
def ziehe(m, n, schritte, geraet):
    # dieselbe Schleife wie D.ziehe, nur mit allem auf dem Geraet
    aq = alpha_quer.to(geraet)
    z = torch.randn(n, D.ENGE, device=geraet)
    stellen = torch.linspace(D.SCHRITTE, 1, schritte).long().tolist()
    for i, t in enumerate(stellen):
        e = m(z, torch.full((n,), t, dtype=torch.long, device=geraet))
        z0 = ((z - (1 - aq[t]).sqrt() * e) / aq[t].sqrt()).clamp(-4, 4)
        e = (z - aq[t].sqrt() * z0) / (1 - aq[t]).sqrt()
        a_vor = aq[stellen[i + 1]] if i + 1 < len(stellen) else aq[0]
        z = a_vor.sqrt() * z0 + (1 - a_vor).sqrt() * e
    return z0


def einheit7_zwoelf(geraet):
    ziehe(basis(geraet), 12, 200, geraet)


def einheit7_regler(geraet):
    m = basis(geraet)
    for s in (2, 5, 10, 25, 50, 200):
        ziehe(m, 8, s, geraet)
        ziehe(m, 64, s, geraet)


def einheit7_lora(geraet):
    # dieselbe Rechnung wie D.trainiere mit LoRA, 1500 Epochen auf der Auswahl
    m = basis(geraet).train()
    for p in m.parameters():
        p.requires_grad_(False)
    m.ein = D.LoRA(m.ein).to(geraet)
    m.aus[1] = D.LoRA(m.aus[1]).to(geraet)
    opt = torch.optim.Adam([p for p in m.parameters() if p.requires_grad], lr=1e-3)
    aq = alpha_quer.to(geraet)
    Zs = Z[torch.tensor(np.load("../data/lora_auswahl.npy"))].to(geraet)
    for _ in range(1500):
        reihe = torch.randperm(len(Zs), device=geraet)
        for anfang in range(0, len(Zs), D.STAPEL):
            z0 = Zs[reihe[anfang:anfang + D.STAPEL]]
            t = torch.randint(1, D.SCHRITTE + 1, (len(z0),), device=geraet)
            zt, rauschen = D.verrausche(z0, t, aq)
            f = nn.functional.mse_loss(m(zt, t), rauschen)
            opt.zero_grad()
            f.backward()
            opt.step()


for g in geraete:
    stoppe("E7  200 Schritte x 12 Bilder", g, einheit7_zwoelf)
    stoppe("E7  Schrittregler, wie im Notebook", g, einheit7_regler)
    stoppe("E7  LoRA-Training der Hausuebung", g, einheit7_lora)

## Ablegen

In [ ]:
ordner = Path.home() / "lastmessung"
ordner.mkdir(exist_ok=True)
stempel = datetime.now().strftime("%Y%m%d-%H%M%S")
datei = ordner / f"{umgebung['rechner']}_{stempel}.json"
datei.write_text(json.dumps(dict(umgebung=umgebung, sekunden=ergebnis, zeit=stempel), indent=1))
print("written:", datei)
print(json.dumps(ergebnis, indent=1))